In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from multiprocessing import Pool, cpu_count, set_start_method

In [ ]:
def calculate_blur(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def process_single_video(args):
    video_path, face_cascade_path = args

    face_cascade = cv2.CascadeClassifier(face_cascade_path)
    cap = cv2.VideoCapture(video_path)

    video_name = os.path.basename(video_path)
    frame_data = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

        metrics = None
        quality_label = 0

        if len(faces) > 0:
            x, y, fw, fh = max(faces, key=lambda x: x[2] * x[3])

            metrics = {
                "face_area": (fw * fh) / (w * h),
                "blur_score": calculate_blur(gray[y:y+fh, x:x+fw])
            }

            if metrics["blur_score"] > 50 and metrics["face_area"] > 0.005:
                quality_label = 1

        frame_data.append({
            "video_name": video_name,
            "frame_idx": frame_idx,
            "quality_label": quality_label,
            "metrics": metrics
        })

        frame_idx += 1

    cap.release()
    return frame_data


def process_meld_parallel(meld_video_dir, output_csv, num_workers=None):
    if num_workers is None:
        num_workers = max(1, cpu_count() - 2)

    video_files = [f for f in os.listdir(meld_video_dir) if f.endswith(".mp4")]
    video_paths = [os.path.join(meld_video_dir, f) for f in video_files]

    face_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    args_list = [(vp, face_cascade_path) for vp in video_paths]

    all_results = []

    with Pool(num_workers) as pool:
        for result in tqdm(
            pool.imap_unordered(process_single_video, args_list),
            total=len(args_list),
            desc="Processing videos"
        ):
            all_results.extend(result)

    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False)

    print(f"✓ Videos: {len(video_files)}")
    print(f"✓ Frames: {len(df)}")
    print(f"✓ Good frames: {df['quality_label'].sum()}")

    return df


if __name__ == "__main__":
    set_start_method("spawn", force=True)

    df = process_meld_parallel(
        meld_video_dir="C:/Users/hp333/Desktop/Multimodel_emotion_detection/test_data",
        output_csv="meld_face_quality.csv",
        num_workers=8
    )


In [ ]:
def extract_good_frames_from_dataset(meld_video_dir, quality_csv, output_dir):
    """
    Extract all good quality frames from MELD videos with bounding boxes from CSV
    """
    # Load quality data
    df = pd.read_csv(quality_csv)
    
    print(f"Total good frames to extract: {len(df)}")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Group by video for efficient processing
    for video_name, group in tqdm(df.groupby('video_name'), desc="Extracting frames"):
        video_path = os.path.join(meld_video_dir, video_name)
        
        if not os.path.exists(video_path):
            print(f"Warning: {video_path} not found, skipping...")
            continue
        
        cap = cv2.VideoCapture(video_path)
        
        video_basename = os.path.splitext(video_name)[0]
        
        # Create subfolder for each video
        video_output_dir = os.path.join(output_dir, video_basename)
        os.makedirs(video_output_dir, exist_ok=True)
        
        # Extract each good frame
        for _, row in group.iterrows():
            frame_idx = row['frame_idx']
            # metrics = row['parsed_metrics']
            # print(metrics)
            # print(type(metrics))
            
            # Set frame position
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            
            if not ret:
                continue
            
            # If you have x, y in metrics, use them:
            x = row['bbox_x']
            y = row['bbox_y']
            w = row['bbox_w']
            h = row['bbox_h']

            # Draw green bounding box
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
            # Save frame with bounding box
            output_path = os.path.join(video_output_dir, f"frame_{frame_idx:04d}.jpg")
            cv2.imwrite(output_path, frame)
        
        cap.release()
    
    print(f"\nExtraction complete! Frames saved to: {output_dir}")
    return output_dir

In [ ]:
# Usage
extracted_frames_dir = extract_good_frames_from_dataset(
    meld_video_dir='C:/Users/hp333/Desktop/Multimodel_emotion_detection/data/MELD.Raw/train/train_splits',
    quality_csv='C:/Users/hp333/Desktop/Multimodel_emotion_detection/meld_face_quality.csv',
    output_dir='meld_frames'
)

In [ ]:
import pandas as pd

df = pd.read_csv("C:/Users/hp333/Desktop/Multimodel_emotion_detection/meld_face_quality.csv")

df.head()

In [ ]:
df.info()